# Pretrain Commutative CNN Encoder

Load the shared unlabeled pretraining dataset and save commutative CNN encoder weights for downstream classification notebooks.

In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

from sklearn.model_selection import train_test_split

from src.ml import (
    CommutativeCNNClassifier,
    CommutativeCNNConfig,
    CommutativeCNNPretrainingConfig,
    LossWeightConfig,
    OptimizationConfig,
    augment_training_tensors_with_rotations,
    load_commutative_cnn_pretraining_config,
    write_commutative_cnn_pretraining_config,
)
from src.tensor_utils import load_unlabeled_tensor_dataset


In [2]:
# User inputs

unlabeled_dataset_path = Path(".dataset_cache/unlabeled_active_high_mid_low_t20_z5_y96_x96_chunks")
pretrained_encoder_path = Path("artifacts/pretrained_commutative_cnn/encoder_state_v5.pt")
validation_fraction = 0.10
train_num_random_rotations = 0
rotation_range_degrees = 0.0

model_config = CommutativeCNNConfig(
    spatial_conv_channels=(16, 32),
    spatial_kernel_size_z=(3, 1),
    spatial_kernel_size_xy=(5, 3),
    spatial_stride_z=(1, 1),
    spatial_stride_xy=(1, 1),
    spatial_pool_kernel_z=(1, 1),
    spatial_pool_kernel_xy=(2, 2),
    spatial_pool_stride_z=(1, 1),
    spatial_pool_stride_xy=(2, 2),
    temporal_st_channels=(48, 64),
    temporal_st_kernel_sizes=(5, 3),
    temporal_ts_channels=(32, 48, 64),
    temporal_ts_kernel_sizes=(7, 5, 3),
    spatial_agg_channels=(32, 64),
    spatial_agg_kernel_size_z=(3, 1),
    spatial_agg_kernel_size_xy=(3, 3),
    spatial_agg_stride_z=(1, 1),
    spatial_agg_stride_xy=(1, 1),
    spatial_agg_pool_kernel_z=(1, 1),
    spatial_agg_pool_kernel_xy=(1, 2),
    spatial_agg_pool_stride_z=(1, 1),
    spatial_agg_pool_stride_xy=(1, 2),
    patch_size_z=1,
    patch_size_xy=16,
    embedding_dim=64,
    probe_local_count=32,
    probe_region_grid=(1, 2, 2),
    probe_time_bins=8,
    probe_frequency_bins=4,
    dropout=0.1,
)
optimization_config = OptimizationConfig(
    batch_size=8,
    epochs=60,
    learning_rate=4e-4,
    weight_decay=1e-4,
    early_stopping_patience=12,
    early_stopping_min_delta=1e-4,
    early_stopping_start_epoch=24,
    early_stopping_monitor="loss",
    early_stopping_smoothing="median",
    early_stopping_smoothing_window=5,
    training_plot_dir="artifacts/pretrained_commutative_cnn/loss_plots",
    training_plot_every_n_epochs=1,
    training_plot_smoothing_window=5,
    scheduler_patience=4,
    scheduler_factor=0.8,
    scheduler_min_lr=1e-6,
    validation_split=0.0,
    random_state=0,
    standardize=True,
    device=None,
    verbose=True,
)
loss_weight_config = LossWeightConfig(
    lambda_cross=0.35,
    lambda_align=0.0,
    cross_warmup_epochs=12,
    cross_ramp_epochs=24,
    probe_mask_probability=0.5,
    probe_alpha_local=1.0,
    probe_alpha_region_time=1.0,
    probe_alpha_derivative=1.0,
    probe_alpha_frequency=1.0,
    probe_alpha_correlation=1.0,
)

pretraining_config = CommutativeCNNPretrainingConfig(
    unlabeled_dataset_path=unlabeled_dataset_path,
    pretrained_encoder_path=pretrained_encoder_path,
    validation_fraction=validation_fraction,
    train_num_random_rotations=train_num_random_rotations,
    rotation_range_degrees=rotation_range_degrees,
    model_config=model_config,
    optimization_config=optimization_config,
    loss_weight_config=loss_weight_config,
)
pretraining_config_path = write_commutative_cnn_pretraining_config(pretraining_config)
pretraining_config = load_commutative_cnn_pretraining_config(pretraining_config_path)
print(f"Loaded commutative CNN pretraining config from {pretraining_config_path}")
pretraining_config


Loaded commutative CNN pretraining config from /run/media/fabrizio/06bb7271-2161-43a4-91f1-98f9b67e9ab2/home/fabrizio/code/ZebraFish/artifacts/pretrained_commutative_cnn/config.yaml


CommutativeCNNPretrainingConfig(unlabeled_dataset_path=PosixPath('.dataset_cache/unlabeled_active_high_mid_low_t20_z5_y96_x96_chunks'), pretrained_encoder_path=PosixPath('artifacts/pretrained_commutative_cnn/encoder_state_v5.pt'), validation_fraction=0.1, train_num_random_rotations=0, rotation_range_degrees=0.0, model_config=CommutativeCNNConfig(spatial_conv_channels=(16, 32), spatial_kernel_size_z=(3, 1), spatial_kernel_size_xy=(5, 3), spatial_stride_z=(1, 1), spatial_stride_xy=(1, 1), spatial_pool_kernel_z=(1, 1), spatial_pool_kernel_xy=(2, 2), spatial_pool_stride_z=(1, 1), spatial_pool_stride_xy=(2, 2), temporal_st_channels=(48, 64), temporal_st_kernel_sizes=(5, 3), temporal_ts_channels=(32, 48, 64), temporal_ts_kernel_sizes=(7, 5, 3), spatial_agg_channels=(32, 64), spatial_agg_kernel_size_z=(3, 1), spatial_agg_kernel_size_xy=(3, 3), spatial_agg_stride_z=(1, 1), spatial_agg_stride_xy=(1, 1), spatial_agg_pool_kernel_z=(1, 1), spatial_agg_pool_kernel_xy=(1, 2), spatial_agg_pool_stride

In [3]:
unlabeled_dataset = load_unlabeled_tensor_dataset(unlabeled_dataset_path)
train_indices, val_indices = train_test_split(
    range(len(unlabeled_dataset["tensors"])),
    test_size=validation_fraction,
    random_state=optimization_config.random_state,
    shuffle=True,
)
X_train_base = unlabeled_dataset["tensors"][train_indices]
X_val = unlabeled_dataset["tensors"][val_indices]
metadata_train_base = unlabeled_dataset["metadata"].iloc[train_indices].reset_index(drop=True)
metadata_val = unlabeled_dataset["metadata"].iloc[val_indices].reset_index(drop=True)
X_train, _, metadata_train = augment_training_tensors_with_rotations(
    X_train_base,
    [0] * len(X_train_base),
    metadata=metadata_train_base,
    num_random_rotations=train_num_random_rotations,
    rotation_range_degrees=rotation_range_degrees,
    random_state=optimization_config.random_state,
)
{
    "all_tensors": unlabeled_dataset["tensors"].shape,
    "all_metadata": unlabeled_dataset["metadata"].shape,
    "train_base_tensors": X_train_base.shape,
    "train_tensors": X_train.shape,
    "val_tensors": X_val.shape,
    "train_base_metadata": metadata_train_base.shape,
    "train_metadata": metadata_train.shape,
    "val_metadata": metadata_val.shape,
}


{'all_tensors': torch.Size([2144, 20, 5, 96, 96]),
 'all_metadata': (2144, 7),
 'train_base_tensors': torch.Size([1929, 20, 5, 96, 96]),
 'train_tensors': torch.Size([1929, 20, 5, 96, 96]),
 'val_tensors': torch.Size([215, 20, 5, 96, 96]),
 'train_base_metadata': (1929, 7),
 'train_metadata': (1929, 7),
 'val_metadata': (215, 7)}

In [ ]:
%%time
model = CommutativeCNNClassifier(
    model_config=model_config,
    optimization_config=optimization_config,
    loss_weight_config=loss_weight_config,
)
model.pretrain(X_train, validation_data=X_val)
pretrained_encoder_path = model.save_pretrained_encoder(pretrained_encoder_path)
pretrained_encoder_path


Commutative CNN model: parameters=170,057 trainable=170,057 size=0.65 MB
cols:
    ep=epoch
    lr=learning_rate
    eta=estimated_time_remaining
    trL=train_loss
    trS=train_self_probe_loss
    trX=train_cross_probe_loss
    trFA=train_feature_alignment_loss
     ep       lr       eta |      trL      trS      trX     trFA |      vaL      vaS      vaX     vaFA


In [ ]:
model.pretrain_history_.tail()